# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide to loading and exploring the FAIR<sup>2</sup> dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the [`mlcroissant`](https://mlcommons.github.io/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

We'll follow the official Croissant specification to examine available record sets.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Initialize the Croissant Dataset object
dataset = mlc.Dataset(croissant_url)

# Print basic info about the dataset
md = dataset.metadata
print(f"Dataset Name: {md.name}\n\nDescription: {md.description}\n\nPublished: {md.date_published}\nVersion: {md.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below, we enumerate record sets with their field and column `@id`s and names for reference.

In [ ]:
# List all available record sets and their details
print("Available Record Sets (by @id and name):\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - @id: {rs['@id']}")
    print(f"    name: {rs.get('name','N/A')}")
    # Fields inside each record set
    if 'field' in rs:
        print("    Fields:")
        for field in rs['field']:
            print(f"      - @id: {field['@id']}")
            print(f"        name: {field.get('name', 'N/A')}")
    if 'column' in rs:
        print("    Columns:")
        for col in rs['column']:
            print(f"      - @id: {col['@id']}")
            print(f"        name: {col.get('name', 'N/A')}")
    print("")

## 3. Data Extraction
Let's load data from the main protein record set (the first/main record set). All Croissant `@id`s are provided in previous output.

You can replace the record set of interest as needed.

In [ ]:
# Select the first record set as an example (often main data table)
if not record_sets:
    raise Exception("No record sets found in this dataset.")
main_record_set_id = record_sets[0]['@id']
print(f"Loading data from record set: {main_record_set_id}")

records = list(dataset.records(record_set=main_record_set_id))
# Convert to DataFrame
df = pd.DataFrame(records)
print("Columns in DataFrame (@id):\n", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by a key attribute based on actual columns. Use the column `@id` as shown above. We'll choose a numeric field, e.g., abundance, peptide count, or molecular weight. Adjust these to the actual `@id` as per your output.

In [ ]:
# Example: use fields called 'molecular_weight' and 'description' if they exist (replace with actual @id if necessary):
from IPython.display import display

# Try to infer some likely candidate column names for numeric analysis
possible_fields = [c for c in df.columns if 'weight' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower()]
if not possible_fields:
    print("No obvious numeric field found: showing dtypes for inspection.")
    print(df.dtypes)
    raise RuntimeError("You may need to adjust the numeric field id below to match dataset columns.")
# Pick a numeric field
numeric_field_id = possible_fields[0]
print(f"Using numeric field: {numeric_field_id}")
if not pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    # Try to coerce
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Set a threshold to filter by (e.g. mean or 10)
threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize selected field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to use a group/categorical field for grouping
group_fields = [c for c in df.columns if 'description' in c.lower() or 'type' in c.lower() or 'group' in c.lower()]
group_field = group_fields[0] if group_fields else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
    print(f"Mean {numeric_field_id} grouped by {group_field}:")
    display(grouped_df.head())
else:
    print("No obvious group/categorical field found to group by.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version, before and after filtering.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.histplot(df[numeric_field_id], kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)

plt.subplot(1, 2, 2)
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, color='salmon')
plt.title(f"Filtered & Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.tight_layout()
plt.show()

# If grouping available, show mean values
if group_field is not None and not grouped_df.empty:
    plt.figure(figsize=(8, 5))
    sns.barplot(y=group_field, x=numeric_field_id, data=grouped_df.head(10))
    plt.title(f"Top 10 {group_field} by mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we:

- Loaded the FAIR<sup>2</sup> mass spectrometry EV dataset via its Croissant schema and explored its metadata.
- Enumerated available record sets and fields using their `@id` for reproducibility.
- Loaded all records from the main data table and inspected key columns.
- Filtered and normalized an important numeric field and performed a basic group analysis.
- Visualized field distributions to gain insights.

You can adapt this template by adjusting the record sets and field `@id`s used in the analysis to meet your research or project requirements.